In [29]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [30]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/UGC-5823_2_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/UGC-5823_2_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     373   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        54   (2094, 964)   float32   


In [31]:
image_cut = image[38:138, 0:1890]
image_error_cut = image_error[38:138, 0:1890]

In [32]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.0188
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [33]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5, params6, params7):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) +
        sersic_1d(x_hr, params6) +
        sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [34]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/PSF/Real_seeing_UGC5823_2.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [35]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, params6, params7, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5, params6, params7)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, params6, params7, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    purple_area = single_integral(x, params6, a, b)
    pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area + purple_area + pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, params6, params7, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area, (purple_area*100)/sum_area, (pink_area*100)/sum_area]     

## Halpha

In [36]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/Halpha_fit.csv", index_col=0)

In [37]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5845402670184596
1.6196890925783567
7.607385737014641
8.829774182337516
3.580725077549781
3.257156078901806
0.6119669696120797
0.6118202310182674
0.8733748290117713
1.244799911385442
1.5610873452593568
1.6169814091628067
3.245987438507009
3.25805144512821
                  Peak 1        Peak 2        Peak 3        Peak 4  \
Blue        9.662352e+01  8.676128e-01  5.465420e-02  1.009840e-01   
Green       2.896486e+00  9.878353e+01  1.990371e-01  3.603152e-01   
Magenta     2.609029e-01  2.130622e-01  9.598728e+01  5.626035e+01   
Orange      0.000000e+00  0.000000e+00  1.407549e-01  2.543606e+01   
Chocolate   7.746807e-40  7.945481e-34  1.103971e-15  3.206132e-10   
Dodgerblue  2.190896e-01  1.357918e-01  3.618271e+00  1.784229e+01   
Deeppink    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   

                  Peak 5        Peak 6        Peak 7  
Blue        6.227521e-02  8.793341e-03  5.233499e-03  
Green       2.285974e-01  3.472820e-02  2.223422e-02  
Magenta     1.21

In [38]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

2.9241931458429855
2.9805269372458447
12.956835394199206
14.905862040497473
6.339671125727471
5.823324280249213
3.1985854134892153
3.146054793456556
1.768213559529291
2.360054151285193
7.091490692104401
7.27031598494059
5.973325500671456
5.993076610060033
                  Peak 1        Peak 2        Peak 3        Peak 4  \
Blue        9.627521e+01  1.034556e+00  6.210468e-02  9.756739e-02   
Green       3.202904e+00  9.855365e+01  2.265718e-01  3.486236e-01   
Magenta     2.838345e-01  2.518087e-01  9.516333e+01  6.494484e+01   
Orange      0.000000e+00  0.000000e+00  4.310603e-01  1.725046e+01   
Chocolate   1.256678e-39  1.585293e-33  6.280788e-15  1.783398e-07   
Dodgerblue  2.380544e-01  1.599818e-01  4.116936e+00  1.735851e+01   
Deeppink    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   

                  Peak 5     Peak 6        Peak 7  
Blue        6.180944e-02   0.009754  5.710351e-03  
Green       2.268283e-01   0.038470  2.423740e-02  
Magenta     1.222425e+01   

In [39]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

4.377071699713828
4.443031912317585
16.75248360543551
18.999996109692905
8.797418015513824
8.195284545921194
4.9004419793246665
4.74226319902337
3.490284512658946
4.170220429993422
9.34018776696749
9.521258426340417
8.558821120144398
8.58226219451963
                  Peak 1        Peak 2        Peak 3     Peak 4        Peak 5  \
Blue        9.500674e+01  1.587881e+00  8.388534e-02   0.090127  5.297241e-02   
Green       4.353762e+00  9.787256e+01  3.090507e-01   0.322559  1.949561e-01   
Magenta     3.493315e-01  3.293173e-01  9.350295e+01  71.396503  1.045289e+01   
Orange      0.000000e+00  0.000000e+00  1.093767e+00  12.033024  7.596290e-04   
Chocolate   6.037684e-39  3.937147e-33  6.098268e-14   0.000148  1.579347e+00   
Dodgerblue  2.901673e-01  2.102435e-01  5.010344e+00  16.157639  8.771907e+01   
Deeppink    0.000000e+00  0.000000e+00  0.000000e+00   0.000000  3.478776e-14   

               Peak 6        Peak 7  
Blue         0.010456  6.753562e-03  
Green        0.041179  2

## HBeta

In [40]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/Hbeta_fit.csv", index_col = 0)

In [41]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.34671252706075234
0.35410111138601486
2.032166312289904
2.121036422140435
1.0080880864510222
0.9929614251211675
0.2313910393718291
0.23144334793960641
0.2139425689641235
0.26672775198407495
0.5887383554076988
0.6010207115456845
1.2308483779684327
1.273995238565502
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.604260e+01  2.673185e+00  1.922629e-01  2.647367e-01   0.118975   
Green       1.586325e+00  9.538291e+01  9.978881e-02  1.175511e-01   0.042688   
Magenta     2.262625e+00  1.814100e+00  9.506441e+01  8.144214e+01  26.789158   
Orange      1.879573e-74  1.174517e-38  2.214049e-02  4.954364e-01   0.000190   
Chocolate   2.545269e-23  1.424615e-21  3.747743e-14  2.135878e-11   0.004288   
Dodgerblue  1.084474e-01  1.298042e-01  4.620917e+00  1.767481e+01  72.807115   
Deeppink    2.873236e-09  4.427418e-08  4.772112e-04  5.323135e-03   0.237586   

                  Peak 6        Peak 7  
Blue        2.249260e-02  1.308198e-02  
Gr

In [42]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)
df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5682306227017349
1.5920245069869248
3.5130719647594546
3.65528238278504
1.8677444389835598
1.8436713028238243
1.2279895668896568
1.2256772325856988
1.1040534613716935
1.2827331217355444
2.7439818165346224
2.785221001604122
2.264966690641508
2.334041373308556
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.552377e+01  3.129795e+00  2.087311e-01  2.518774e-01   0.116286   
Green       1.846393e+00  9.461457e+01  1.089832e-01  1.127367e-01   0.041908   
Magenta     2.509146e+00  2.104806e+00  9.461984e+01  8.228741e+01  26.292936   
Orange      1.139501e-69  7.670118e-37  3.261906e-02  3.829060e-01   0.000840   
Chocolate   3.089883e-23  1.784500e-21  6.513325e-14  3.562297e-10   0.002954   
Dodgerblue  1.206881e-01  1.508339e-01  5.029278e+00  1.695934e+01  73.262914   
Deeppink    3.369201e-09  5.319957e-08  5.488104e-04  5.731427e-03   0.282162   

                  Peak 6        Peak 7  
Blue        2.432586e-02  1.427797e-02  
Green   

In [43]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


2.0195322742746282
2.0436544367863956
4.677188716631506
4.840282897556143
2.8120783470813673
2.7832085715831223
1.8288533103717097
1.8206981545619518
1.60208413996104
1.7886824996283477
3.678324486989002
3.7291245389680605
3.6307656414858043
3.7156456544880174
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.500630e+01  4.576674e+00  2.595930e-01  2.393136e-01   0.113265   
Green       2.124231e+00  9.253428e+01  1.406714e-01  1.080531e-01   0.041017   
Magenta     2.737302e+00  2.697255e+00  9.392481e+01  8.308198e+01  25.732833   
Orange      4.325740e-67  7.096749e-35  4.584124e-02  3.011189e-01   0.002199   
Chocolate   3.708105e-23  2.368706e-21  1.389572e-13  3.850167e-08   0.002174   
Dodgerblue  1.321670e-01  1.917867e-01  5.628453e+00  1.626334e+01  73.772086   
Deeppink    3.900105e-09  6.748728e-08  6.317264e-04  6.193679e-03   0.336426   

                  Peak 6        Peak 7  
Blue        2.563067e-02  1.649035e-02  
Green   

## NII

In [44]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/NII_fit.csv", index_col = 0)

In [45]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.25696412167620447
0.2646328792934911
2.0157542256165346
1.9843172322585425
0.9912693929493708
0.9241854868970317
0.4324205035648413
0.48639154465257955
0.5279326851729219
0.5385877735585534
0.4007426419535258
0.40856064104970086
0.8136886586713654
0.7545297008106564
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.620311e+01  4.578158e+00  1.943337e+00  3.600258e+00   2.065850   
Green       1.669909e+00  9.401944e+01  6.385523e-02  6.879948e-02   0.016760   
Magenta     9.036149e-01  7.367674e-01  8.973899e+01  6.187598e+01  12.007066   
Orange      9.000126e-05  7.319413e-05  3.951739e-02  5.876575e+00   0.013065   
Chocolate   1.070546e-12  2.603171e-10  1.976353e-03  1.223091e-01  26.936613   
Dodgerblue  1.223278e+00  6.655639e-01  8.212324e+00  2.845608e+01  58.960643   
Deeppink    1.649496e-56  1.083151e-43  2.110817e-18  9.951672e-14   0.000003   

               Peak 6     Peak 7  
Blue         1.071376   0.870620  
Green       

In [46]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.1749776144647122
1.199954210018723
3.484478289865498
3.4352625612579013
1.8217226417737014
1.7158350132001956
0.880217081553549
0.9648300410725066
1.2301472397585917
1.2484763453700618
1.8715410112103368
1.8948309760177087
1.4899981557552873
1.396556068035916
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.572860e+01  5.320122e+00  2.117491e+00  3.541200e+00   2.262450   
Green       1.936586e+00  9.305189e+01  7.047588e-02  6.838683e-02   0.019458   
Magenta     9.936616e-01  8.565043e-01  8.876405e+01  6.345581e+01  13.983615   
Orange      9.895619e-05  8.516229e-05  7.376396e-02  4.682370e+00   0.019111   
Chocolate   1.411801e-12  3.382436e-10  2.599630e-03  1.528405e-01  23.052143   
Dodgerblue  1.341057e+00  7.714014e-01  8.971616e+00  2.809939e+01  60.663213   
Deeppink    2.512560e-55  5.168663e-43  8.621853e-18  3.565809e-13   0.000009   

               Peak 6     Peak 7  
Blue         1.148693   0.951573  
Green        0.0054

In [47]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] +3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

1.5307990068182649
1.5563560691920508
4.641075401208112
4.583579327892702
2.9666019641848633
2.8585849126793295
1.5414176899483565
1.6323680084817243
1.6803244683968137
1.6985255335806924
2.5108518821953205
2.5274344679342
2.5386818164766227
2.4339226804952383
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.526125e+01  7.361454e+00  2.613448e+00  3.476684e+00   2.323015   
Green       2.219212e+00  9.055743e+01  9.155333e-02  6.560991e-02   0.020301   
Magenta     1.074335e+00  1.090300e+00  8.569543e+01  6.310153e+01  14.644200   
Orange      1.069748e-04  1.084051e-04  4.036205e-01  3.163738e+00   0.024151   
Chocolate   1.833618e-12  4.684822e-10  6.257060e-03  4.156640e-01  19.252693   
Dodgerblue  1.445093e+00  9.907054e-01  1.118969e+01  2.977677e+01  63.735597   
Deeppink    1.611738e-54  3.454254e-42  2.832098e-16  1.108224e-11   0.000044   

               Peak 6     Peak 7  
Blue         1.200588   1.120196  
Green        0.00576

## SII

In [48]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/SII_fit.csv", index_col = 0)

In [49]:
fit

,Component 1,Component 2,Component 3,Component 4,Component 5,Component 6,Component 7
I_e,0.0182,0.0547,0.0290,3.9481,0.0449,0.0226,0.6328
r_e,20.9444,5.4340,16.5770,0.0000,0.0636,33.7633,0.2955
n,1.7733,2.1194,1.5224,4.1999,2.6501,1.7847,0.3631
x_0,14.2773,28.8410,57.0877,61.5756,71.2236,77.4035,83.1078


In [50]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.5446833989368077
0.5522166055806249
1.3514681998454336
1.3296154363519985
0.7001576002389921
0.6896955607852601
0.22025930465530652
0.22037271246172732
0.41394446250054234
0.4846895910625101
0.3902682495349613
0.39807309822721343
0.9051194736756893
0.7633195348267039
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.491921e+01  4.882064e+00  1.062385e+00  1.275883e+00   0.779424   
Green       1.974135e+00  9.208708e+01  1.629271e-01  1.392440e-01   0.045930   
Magenta     9.996294e-01  1.406702e+00  8.435438e+01  6.701636e+01  19.160636   
Orange      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   
Chocolate   2.236936e-25  1.106339e-22  1.614520e-13  1.125820e-10   0.290406   
Dodgerblue  2.107026e+00  1.624159e+00  1.442030e+01  3.156851e+01  79.723605   
Deeppink    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   

                  Peak 6        Peak 7  
Blue        2.893282e-01  1.868364e-01  

C:\Users\ISAFA\AppData\Local\Temp\ipykernel_11532\3564233926.py:10: RuntimeWarning: divide by zero encountered in divide
  return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))


In [51]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

C:\Users\ISAFA\AppData\Local\Temp\ipykernel_11532\3564233926.py:10: RuntimeWarning: divide by zero encountered in divide
  return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))


1.0283661720037702
1.0404346466195575
2.380423435941615
2.345962615475024
1.3312752143459132
1.3146490291935813
1.1376738536665005
1.1375581439180433
0.8363868847819506
0.9485505033824435
1.8207935723691775
1.8413711278039995
1.606466603444363
1.3846214776275385
                  Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue        9.456019e+01  5.581220e+00  1.120110e+00  1.239930e+00   0.772891   
Green       2.143167e+00  9.096755e+01  1.733804e-01  1.372405e-01   0.045800   
Magenta     1.062459e+00  1.604684e+00  8.349255e+01  6.784247e+01  19.120942   
Orange      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   
Chocolate   2.599877e-25  1.439223e-22  3.071875e-13  1.308345e-09   0.228027   
Dodgerblue  2.234188e+00  1.846546e+00  1.521396e+01  3.078036e+01  79.832340   
Deeppink    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   0.000000   

               Peak 6        Peak 7  
Blue         0.310995  2.108516e-01  
Green       

In [52]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

1.6113477370785563
1.6255302701883492
3.283162202926173
3.242357381951509
2.1171425321564508
2.098235973132534
1.6427444713720392
1.6405008992853198
1.5734899120889492
1.7018769551194377
2.4440371842905506
2.4480609404034497
2.2764151934752466
2.029104635211652
                  Peak 1        Peak 2        Peak 3        Peak 4  \
Blue        9.334948e+01  7.604227e+00  1.201708e+00  1.207339e+00   
Green       2.851573e+00  8.812707e+01  1.832572e-01  1.357037e-01   
Magenta     1.245235e+00  1.971011e+00  8.114548e+01  6.857623e+01   
Orange      0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
Chocolate   4.613362e-25  1.981920e-22  2.864463e-12  2.055970e-08   
Dodgerblue  2.553711e+00  2.297692e+00  1.746956e+01  3.008072e+01   
Deeppink    0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   

                   Peak 5     Peak 6        Peak 7  
Blue         7.024881e-01   0.325419  2.548034e-01  
Green        4.102978e-02   0.013542  7.782898e-03  
Magenta      1.716

C:\Users\ISAFA\AppData\Local\Temp\ipykernel_11532\3564233926.py:10: RuntimeWarning: divide by zero encountered in divide
  return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))


## OIII

In [53]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/UGC5823/UGC5823_2/FORS2.2022-01-07T07_40_46.357/Fit/OIII_fit.csv", index_col = 0)

In [54]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -sigma, fit['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -sigma, fit['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -sigma, fit['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -sigma, fit['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -sigma, fit['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -sigma, fit['Component 6'].iloc[3] + sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -sigma, fit['Component 7'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

0.9516432306232041
0.9626361042869919
0.831228751190903
1.0354554861782548
1.0934349308697515
1.125677519533454
0.5855098403314919
0.5861018254536565
0.3995969058974118
0.399131537985656
1.2892015148228928
1.2832548513341235
0.4202569891459641
0.868205843105359
                  Peak 1        Peak 2        Peak 3        Peak 4  \
Blue        9.772883e+01  6.115766e+00  1.076505e+00  1.687881e+00   
Green       1.732692e+00  9.287806e+01  2.060808e-01  2.727279e-01   
Magenta     3.836685e-01  6.950497e-01  9.101313e+01  7.790340e+01   
Orange      4.477134e-31  7.355761e-29  2.091779e-10  3.147248e-09   
Chocolate   1.513913e-24  2.688110e-23  1.957543e-17  1.531121e-15   
Dodgerblue  1.548127e-01  3.111238e-01  7.704287e+00  2.013599e+01   
Deeppink    8.223803e-23  3.598902e-20  9.872575e-13  1.790981e-11   

                  Peak 5        Peak 6        Peak 7  
Blue        1.593533e+00  2.981106e-01  3.691209e-01  
Green       1.748097e-01  2.211669e-02  2.351871e-02  
Magenta     

In [55]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -2*sigma, fit['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -2*sigma, fit['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -2*sigma, fit['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -2*sigma, fit['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -2*sigma, fit['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -2*sigma, fit['Component 6'].iloc[3] + 2*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -2*sigma, fit['Component 7'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

1.770983763574818
1.788692391310157
3.3923749842595026
4.073964460434337
2.002397319631108
2.0536544063274436
1.4295057903780715
1.4343625032157334
1.05897693289442
1.0575730151744158
2.4616355328400665
2.487076305611705
2.053496163487046
3.5707143571311892
                  Peak 1        Peak 2        Peak 3        Peak 4  \
Blue        9.751840e+01  7.595100e+00  1.178164e+00  1.685367e+00   
Green       1.901040e+00  9.116152e+01  2.269513e-01  2.675537e-01   
Magenta     4.135689e-01  8.592537e-01  9.012445e+01  7.592664e+01   
Orange      5.099138e-31  1.103294e-28  3.857944e-10  2.181825e-09   
Chocolate   1.665091e-24  3.542880e-23  3.391572e-17  1.764232e-11   
Dodgerblue  1.669933e-01  3.841279e-01  8.470431e+00  2.212044e+01   
Deeppink    9.691281e-23  5.465188e-20  1.391436e-12  4.568073e-11   

                  Peak 5        Peak 6        Peak 7  
Blue        1.468377e+00  3.126775e-01  3.785885e-01  
Green       1.586479e-01  2.325917e-02  2.422488e-02  
Magenta     1.78

In [56]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 1'].iloc[3] -3*sigma, fit['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 2'].iloc[3] -3*sigma, fit['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 3'].iloc[3] -3*sigma, fit['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 4'].iloc[3] -3*sigma, fit['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 5'].iloc[3] -3*sigma, fit['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 6'].iloc[3] -3*sigma, fit['Component 6'].iloc[3] + 3*sigma)

data7 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit['Component 7'], fit['Component 7'].iloc[3] -3*sigma, fit['Component 7'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1), np.array(data7).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(7)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Dodgerblue', 'Deeppink'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

2.6969520438145778
2.7180456007799796
4.042106623902092
4.7514136759796255
2.9444756792444893
3.002011632659543
2.1482357027259495
2.165892294416511
1.5172327458652886
1.5151571031141535
3.9146698240845446
4.088110101092285
2.867208679231159
4.455057863364566
                  Peak 1        Peak 2        Peak 3        Peak 4  \
Blue        9.675267e+01  9.058973e+00  1.453017e+00  1.576195e+00   
Green       2.548425e+00  8.946630e+01  2.935684e-01  2.528513e-01   
Magenta     4.967068e-01  1.019585e+00  8.853706e+01  7.725559e+01   
Orange      8.133615e-31  1.596920e-28  5.668714e-10  1.553163e-09   
Chocolate   2.265704e-24  4.501035e-23  7.793119e-17  1.660982e-10   
Dodgerblue  2.022016e-01  4.551441e-01  9.716352e+00  2.091537e+01   
Deeppink    1.763317e-22  7.972353e-20  2.122777e-12  7.101638e-11   

                  Peak 5        Peak 6        Peak 7  
Blue        1.439504e+00  3.538672e-01  3.805662e-01  
Green       1.566836e-01  2.699396e-02  2.446495e-02  
Magenta     1.